# Explicit-State CTL Model Checking
**Introduction to Software Verification (2360342) · Lectures 3–4**

We build a complete explicit-state CTL model checker:

1. the formula as a **graph** of its sub-formulas
2. a **post-order** traversal of that graph (sub-formulas first)
3. the **top-level** model-checking algorithm, which calls one procedure per operator
4. the procedure for **EX**, run on a small example
5. the **microwave** example
6. **Exercise:** implement **EU** and **EG**
7. **Fairness**

Everything that is printed or drawn is controlled by `mc.settings(trace=..., draw=..., summary=...)`.

In [ ]:
# Setup: works in GitHub Codespaces, Jupyter and Google Colab
import os
try:
    import igraph
except ImportError:                                  # e.g. on Colab: install igraph
    %pip install -q igraph
if not os.path.exists('ctlmc.py'):                   # e.g. on Colab: fetch the helpers
    !wget -q https://raw.githubusercontent.com/TechnionFV/intro-fv-exercises/main/01-explicit-ctl-mc/ctlmc.py
%matplotlib inline
import ctlmc as mc
mc.settings(trace=True, draw=True, summary=True)

## 1. The formula as a graph

A CTL formula is parsed and then rewritten into the basis **¬, ∧, ∨, EX, EU, EG**
(the identities on the *Expressiveness* slide). The result is a graph:

* one vertex per **distinct** sub-formula, labeled by its operator (`op`)
* an edge from every sub-formula to its operands, kept in order in `children` (the order matters for EU)
* a sub-formula that occurs twice is a **single** vertex, so the graph is a DAG

In [ ]:
phi = 'AG(Start -> AF Heat)'
ast = mc.parse(phi)
print('parsed     :', mc.to_str(ast))
print('normalized :', mc.to_str(mc.normalize(ast)))

F = mc.formula_graph(phi)
mc.draw_formula(F)

In [ ]:
for v in F.vs:
    print(f"vertex {v.index:2}   op={v['op']:5}   children={str(v['children']):8}   {v['label']}")
print('root:', F['root'])

**Sharing.** In `EX p & EG EX p` the sub-formula `EX p` occurs twice but is a single vertex.
It is checked only once, which is why the algorithm needs O(|φ|) procedure calls.

In [ ]:
mc.draw_formula(mc.formula_graph('EX p & EG EX p'))

## 2. Post-order traversal

The algorithm processes a sub-formula only **after all its sub-formulas** have been processed.
A post-order DFS on the formula graph gives exactly this order (the leaves first, the root last).

In [ ]:
def postorder(F):
    visited, order = set(), []
    def visit(v):
        if v in visited:                      # shared sub-formula: visited already
            return
        visited.add(v)
        for c in F.vs[v]['children']:         # first the operands ...
            visit(c)
        print(f"visit {v:2}:  {F.vs[v]['label']}")   # ... then the vertex itself
        order.append(v)
    visit(F['root'])
    return order

order = postorder(F)

In [ ]:
# the same order, drawn step by step: maroon = visited, green frame = current
for k, v in enumerate(order):
    mc.draw_formula(F, done=order[:k], current=v, full=False, title=f"step {k + 1}: {F.vs[v]['label']}")

## 3. The top-level algorithm

`model_check` runs the post-order traversal. At every vertex it calls the procedure for that
vertex's operator, which receives the sets computed for the operands:

`sat[v] = procedure(op)(M, node, sat[child_1], sat[child_2], ...)`

`sat[v]` is the set of states ⟦ψ⟧ in which the sub-formula ψ of vertex `v` holds.
Finally, **M ⊨ φ iff S₀ ⊆ ⟦φ⟧**.

In [ ]:
def model_check(M, F, procedure):
    sat = {}                                          # vertex of F  ->  [[sub-formula]]
    def visit(v):
        if v in sat:                                  # shared sub-formula: checked already
            return
        for c in F.vs[v]['children']:                 # first all sub-formulas ...
            visit(c)
        node = F.vs[v]
        mc.log(f"\n=== checking  {node['label']}   (op = {node['op']})")
        sat[v] = procedure(node['op'])(M, node, *[sat[c] for c in node['children']])   # ... then the operator
        mc.log(f"=== [[ {node['label']} ]] = {mc.fmt(sat[v])}")
        if mc.DRAW_SUMMARY:
            mc.draw_step(M, F, v, sat)
    visit(F['root'])
    return sat

def check(M, phi, procedure):
    """Parse phi, build its graph, run the model checker and report  M |= phi ?"""
    F = mc.formula_graph(phi)
    print(f"formula: {phi}\nchecked: {F['formula']}")
    sat = model_check(M, F, procedure)
    S = sat[F['root']]
    holds = M['init'] <= S
    print(f"\nM |= {phi} ?   {holds}      (S0 = {mc.fmt(M['init'])},  [[phi]] = {mc.fmt(S)})")
    return holds

The procedures for atomic propositions and the Boolean operators only look at each state once.

In [ ]:
def check_AP(M, node):                 # p ∈ label(s)  iff  p ∈ L(s)
    return {s for s in mc.states(M) if node['ap'] in M.vs[s]['props']}

def check_true(M, node):
    return mc.states(M)

def check_not(M, node, S1):            # S1 = [[f1]]
    return mc.states(M) - S1

def check_and(M, node, S1, S2):
    return S1 & S2

def check_or(M, node, S1, S2):
    return S1 | S2

## 4. The procedure for EX

Like `CheckEX` on the slides: start from the states where f₁ holds and go **backwards**,
so every predecessor is labeled with EX f₁.
`mc.show(...)` prints each step and draws it: maroon states are labeled, a green ring means the state is in T.

In [ ]:
def check_EX(M, node, S1):
    label = set()
    T = list(S1)                                         # T := { t | f1 ∈ label(t) }
    mc.show(M, label, T, msg='EX: T := [[f1]]')
    while T:                                             # while T ≠ ∅
        t = T.pop()                                      #   choose t ∈ T;  T := T \ {t}
        for s in M.predecessors(t):                      #   forall s s.t. R(s, t)
            if s not in label:                           #     if EX f1 ∉ label(s)
                label.add(s)                             #       label(s) := label(s) ∪ {EX f1}
                mc.show(M, label, T, current=s, msg=f'EX: {s} -> {t}, label {s}')
    return label

**EU and EG are still empty.** They return the empty set, so they do no harm as long as the formula does not use them.

In [ ]:
def check_EU(M, node, S1, S2):
    return set()          # not implemented yet (Part 6)

def check_EG(M, node, S1):
    return set()          # not implemented yet (Part 6)

**Dispatch.** `procedure(op)` returns the procedure for an operator.
It looks the functions up when it is called, so redefining `check_EU` or `check_EG` later takes effect immediately.

In [ ]:
def procedure(op):
    return {'AP': check_AP, 'true': check_true, 'not': check_not, 'and': check_and, 'or': check_or,
            'EX': check_EX, 'EU': check_EU, 'EG': check_EG}[op]

### A small example: only AP, ¬, ∧ and EX

In [ ]:
M0 = mc.toy_model()
mc.draw_model(M0, title='Toy model  (green arrow = initial state)')

In [ ]:
check(M0, 'p & EX (q & !p)', procedure)

In [ ]:
# AX is rewritten as !EX!, so this also uses only AP, ¬ and EX
check(M0, 'AX q', procedure)

## 5. The microwave example

States are numbered 0–6 here (1–7 on the slides). The property is **AG(Start → AF Heat)**,
which is checked as **¬E(true U (Start ∧ EG ¬Heat))**.

In [ ]:
M = mc.microwave()
mc.draw_model(M, title='Microwave')

In [ ]:
mc.draw_formula(mc.formula_graph('AG(Start -> AF Heat)'))

Running it now gives a **wrong** answer: with EU and EG empty, the checker claims the property holds.

In [ ]:
mc.settings(draw=False)        # draw only the summary after each sub-formula
check(M, 'AG(Start -> AF Heat)', procedure)
mc.settings(draw=True)

## 6. Exercise: implement EU and EG

Fill in the two procedures below, following the slides. Use `M.predecessors(s)` to go backwards, and call
`mc.show(M, label, T, current=..., msg=...)` whenever you label a state, to print and draw the step.

**CheckEU(f₁, f₂)**
```
T := { s | f2 ∈ label(s) }
forall s ∈ T do label(s) := label(s) ∪ { E(f1 U f2) }
while T ≠ ∅ do
    choose s ∈ T;  T := T \ {s}
    forall t s.t. R(t, s) do
        if E(f1 U f2) ∉ label(t) and f1 ∈ label(t) then
            label(t) := label(t) ∪ { E(f1 U f2) };   T := T ∪ {t}
```

**CheckEG(f₁)**
```
S' := { s | f1 ∈ label(s) }
MSCC := { C | C is a nontrivial MSCC of M' }          ← use mc.nontrivial_sccs(M, S1)
T := ⋃ C∈MSCC { s | s ∈ C };   label all of T with EG f1
while T ≠ ∅ do
    choose s ∈ T;  T := T \ {s}
    forall t ∈ S' s.t. R'(t, s) do
        if EG f1 ∉ label(t) then  label(t) := label(t) ∪ { EG f1 };   T := T ∪ {t}
```

In [ ]:
def check_EU(M, node, S1, S2):
    """[[E(f1 U f2)]], given S1 = [[f1]] and S2 = [[f2]]."""
    label = set()
    # IMPLEMENT
    return label

In [ ]:
def check_EG(M, node, S1):
    """[[EG f1]], given S1 = [[f1]]."""
    label = set()
    # IMPLEMENT  (hint: mc.nontrivial_sccs(M, S1) returns the nontrivial MSCCs of M' as a list of sets)
    return label

**Self-check.** Compares your procedures with the expected sets on the microwave (no printing or drawing).

In [ ]:
def self_check():
    tests = [('E(Start U Close)', {1, 2, 3, 4, 5, 6}),
             ('E(Close U Heat)', {2, 3, 4, 5, 6}),
             ('E(true U Error)', {0, 1, 2, 3, 4, 5, 6}),
             ('EG !Heat', {0, 1, 2, 4}),
             ('EG Start', {1, 4}),
             ('EG Close', {2, 3, 4, 5, 6}),
             ('EG (Close & !Start)', {3}),
             ('Start & EG !Heat', {1, 4}),
             ('E(true U (Start & EG !Heat))', {0, 1, 2, 3, 4, 5, 6})]
    saved = (mc.TRACE, mc.DRAW, mc.DRAW_SUMMARY)
    mc.settings(trace=False, draw=False, summary=False)
    M = mc.microwave()
    try:
        for phi, expected in tests:
            F = mc.formula_graph(phi)
            got = model_check(M, F, procedure)[F['root']]
            print(('✓ ' if got == expected else '✗ ') + f"[[ {phi} ]] = {mc.fmt(got)}" +
                  ('' if got == expected else f"   expected {mc.fmt(expected)}"))
    finally:
        mc.settings(*saved)

self_check()

Now run the microwave property again; this time every step of EU and EG is printed and drawn.

In [ ]:
check(M, 'AG(Start -> AF Heat)', procedure)

## 7. Fairness

Fairness constraint **F = { Start ∧ Close ∧ ¬Error }**, i.e. a fair path visits states 5 or 6 infinitely often.
Following Lecture 4:

* **EG** becomes **E_F G**: only nontrivial MSCCs of M' that intersect every P ∈ F count
* compute **fair := E_F G true**, the states from which a fair path starts
* **EX f₁** becomes **EX(f₁ ∧ fair)**, and **E(f₁ U f₂)** becomes **E(f₁ U (f₂ ∧ fair))**

The model checker itself does not change; only the dispatch does.

In [ ]:
FAIRNESS = ['Start & Close & !Error']

def evaluate(M, phi, procedure):
    """[[phi]] without printing or drawing."""
    saved = (mc.TRACE, mc.DRAW, mc.DRAW_SUMMARY)
    mc.settings(trace=False, draw=False, summary=False)
    try:
        F = mc.formula_graph(phi)
        return model_check(M, F, procedure)[F['root']]
    finally:
        mc.settings(*saved)

P_sets = [evaluate(M, p, procedure) for p in FAIRNESS]
print('fairness sets:', [mc.fmt(P) for P in P_sets])

In [ ]:
def check_fair_EG(M, node, S1):
    """[[E_F G f1]]: like EG, but only *fair* MSCCs (C ∩ P ≠ ∅ for every P ∈ F) count."""
    return mc.fair_EG(M, S1, P_sets)

In [ ]:
mc.settings(trace=False, draw=False)
FAIR = check_fair_EG(M, None, mc.states(M))          # [[E_F G true]]
mc.settings(trace=True, draw=True)
print('fair =', mc.fmt(FAIR))

def procedure_fair(op):
    if op == 'EG':
        return check_fair_EG
    if op == 'EX':
        return lambda M, node, S1: check_EX(M, node, S1 & FAIR)
    if op == 'EU':
        return lambda M, node, S1, S2: check_EU(M, node, S1, S2 & FAIR)
    return procedure(op)

In [ ]:
check(M, 'AG(Start -> AF Heat)', procedure_fair)

Same model, same model checker, one change in the dispatch, and the answer flips:
the only cycle that avoids **Heat** (states 0, 1, 2, 4) never visits 5 or 6, so it is **not fair**.